# Fine Tuning de Imagenes con Hugging Face Transformers

El fine tuning de una red neuronal consiste en ajustar un modelo preentrenado con un conjunto de datos mas pequeno y especifico para nuestra tarea.

En este notebook vamos a entrenar un clasificador de imagenes (bananas maduras vs normales) usando `transformers` de Hugging Face, de forma completamente programatica y reproducible.

## Paso 1: Instalar dependencias

Instalaremos librerias para entrenamiento, manejo de datasets y evaluacion.

> Si estas en Google Colab, ejecuta esta celda y reinicia el entorno si te lo pide.

In [ ]:
!pip install -q -U transformers datasets accelerate evaluate pillow torchvision requests

## Paso 2: Descargar y preparar carpetas `train/` y `valid/`

Vamos a crear la estructura esperada por `load_dataset("imagefolder")`:

- `data/bananas_hf/train/normal`
- `data/bananas_hf/train/madura`
- `data/bananas_hf/valid/normal`
- `data/bananas_hf/valid/madura`

In [ ]:
from pathlib import Path
import requests

base_url = "https://github.com/amiune/freecodingtour/raw/main/cursos/espanol/deeplearning/data/bananas/"
root_dir = Path("data/bananas_hf")

for split in ["train", "valid"]:
    for label in ["normal", "madura"]:
        (root_dir / split / label).mkdir(parents=True, exist_ok=True)

# Dejamos 4 imagenes por clase para train y 1 para valid
for label in ["normal", "madura"]:
    for i in range(1, 6):
        filename = f"{label}{i}.jpeg"
        split = "valid" if i == 5 else "train"
        local_path = root_dir / split / label / filename
        if not local_path.exists():
            content = requests.get(base_url + filename, timeout=20)
            content.raise_for_status()
            local_path.write_bytes(content.content)

print("Dataset listo en:", root_dir)

## Paso 3: Cargar dataset desde carpetas

Ahora usamos `datasets` con `imagefolder`, tal como en tu ejemplo base.

In [ ]:
from datasets import load_dataset

ds = load_dataset("imagefolder", data_dir=str(root_dir))

labels = ds["train"].features["label"].names
id2label = {i: l for i, l in enumerate(labels)}
label2id = {l: i for i, l in enumerate(labels)}

id2label, label2id

## Paso 4: Procesador, transform y modelo

Elegimos un checkpoint preentrenado y aplicamos `with_transform` para procesar imagenes on-the-fly.

In [ ]:
import numpy as np
from transformers import AutoImageProcessor, AutoModelForImageClassification

checkpoint = "google/vit-base-patch16-224-in21k"
processor = AutoImageProcessor.from_pretrained(checkpoint)


def transform(batch):
    images = [img.convert("RGB") for img in batch["image"]]
    inputs = processor(images=images, return_tensors="pt")
    inputs["labels"] = batch["label"]
    return inputs


ds = ds.with_transform(transform)

model = AutoModelForImageClassification.from_pretrained(
    checkpoint,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

## Paso 5: Metricas, entrenamiento y evaluacion

Definimos accuracy, entrenamos con `Trainer` y evaluamos en validacion.

In [ ]:
import torch
import evaluate
from transformers import TrainingArguments, Trainer

acc = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, y = eval_pred
    preds = np.argmax(logits, axis=1)
    return acc.compute(predictions=preds, references=y)

args = TrainingArguments(
    output_dir="vit-finetuned",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    num_train_epochs=3,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    remove_unused_columns=False,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds.get("valid", ds.get("test")),
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.evaluate()

## Paso 6: Evaluar el modelo

Calculamos metricas sobre el conjunto de validacion.

In [ ]:
# Ya evaluamos al final de la celda de entrenamiento.
# Si quieres repetir evaluacion:
metricas = trainer.evaluate()
metricas

## Paso 7: Probar inferencia con una imagen nueva

Ahora hacemos una prediccion sobre una imagen que no usamos para entrenar.

In [ ]:
from transformers import pipeline

test_url = "https://raw.githubusercontent.com/amiune/freecodingtour/main/cursos/espanol/deeplearning/data/bananas/banana.jpg"
test_path = root_dir / "banana_test.jpg"

if not test_path.exists():
    content = requests.get(test_url, timeout=20)
    content.raise_for_status()
    test_path.write_bytes(content.content)

clasificador = pipeline(
    "image-classification",
    model=trainer.model,
    image_processor=processor,
    top_k=2,
)

clasificador(str(test_path))

## Paso 8: Guardar modelo y processor

Guardamos ambos artefactos para reutilizar luego en inferencia.

In [ ]:
save_dir = "vit-finetuned"
trainer.save_model(save_dir)
processor.save_pretrained(save_dir)

clasificador_cargado = pipeline(
    "image-classification",
    model=save_dir,
    image_processor=save_dir,
    top_k=2,
)

clasificador_cargado(str(test_path))

## Recomendaciones para mejorar resultados

- Aumentar la cantidad y variedad de imagenes por clase.
- Probar otros checkpoints de vision (por ejemplo, ConvNeXt, Swin o DeiT).
- Ajustar `learning_rate`, `num_train_epochs` y `batch_size`.
- Entrenar en GPU para acelerar el fine tuning.
- Revisar errores con predicciones y probabilidades para detectar casos ambiguos.

# Fin: [Volver al contenido del curso](https://www.freecodingtour.com/cursos/espanol/deeplearning/deeplearning.html)